# Example notebook

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from mudata import AnnData, MuData
import perturbvi
import pandas as pd
import pyro
import torch

perturbvi.__version__

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Global seed set to 0


'0.0.1'

In [3]:
rna_key = "rna"
perturb_key = "grna"
n_cells = 1000

total_rna = pd.DataFrame({'lib_size':np.random.lognormal(10, 1, size=(n_cells))})
rna_counts = np.random.negative_binomial(100, 0.9, size=(n_cells, 10))
rna_adata = AnnData(rna_counts, obs=total_rna, dtype=np.float64)

rna_adata.X

/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


array([[ 3., 21., 11., ...,  8., 10., 12.],
       [12., 12., 11., ...,  9., 12.,  6.],
       [14., 12., 16., ..., 16.,  8.,  9.],
       ...,
       [ 8., 19.,  4., ...,  7., 10., 13.],
       [14., 10., 13., ..., 14., 11., 14.],
       [10., 14., 16., ...,  6., 10., 19.]])

In [4]:
perturb_adata = AnnData(np.random.binomial(1, 0.5, size=(n_cells, 5)), dtype=np.float64)
perturb_adata.var_names = 'guide' + perturb_adata.var_names
mdata = MuData({rna_key: rna_adata, perturb_key: perturb_adata})
mdata

MuData object with n_obs × n_vars = 1000 × 15
  2 modalities
    rna:	1000 x 10
      obs:	'lib_size'
    grna:	1000 x 5

In [5]:
mdata.var_names

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'guide0', 'guide1',
       'guide2', 'guide3', 'guide4'],
      dtype='object')

In [6]:
perturbvi.PERTURBVI.setup_mudata(
    mdata,
    size_factor_key = 'lib_size',
    modalities={
        "rna_layer": rna_key,
        "perturbation_layer": perturb_key,
    },
)

model = perturbvi.PERTURBVI(mdata)
model.summary_stats

n_batch: 1
n_cells: 1000
n_extra_categorical_covs: 0
n_extra_continuous_covs: 0
n_perturbations: 5
n_vars: 10

In [7]:
model = perturbvi.PERTURBVI(mdata)
# pyro.render_model(model.module.model, (torch.tensor(rna_counts),))

In [8]:
model.train(max_epochs=100, train_size=1, lr=0.1)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/configuration_validator.py:106: UserWarning: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
  rank_zero_warn("You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.")
/Users/ljb80/Projects/perturbvi/.hatch/perturbvi/lib/python3.10/site-packages/pytorch_lightning/trainer/trainer.py:1609: PossibleUserWarning: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower 

Epoch 100/100: 100%|██████████| 100/100 [00:04<00:00, 21.86it/s, v_num=1, elbo_train=3.91e+4]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████| 100/100 [00:04<00:00, 21.41it/s, v_num=1, elbo_train=3.91e+4]


In [9]:
for k,v in pyro.get_param_store().items():
    print(k, v.detach().cpu().numpy())

log_var_mean.mu [-6.1570983 -6.1761203 -6.236557  -6.191307  -6.2033277 -6.2584667
 -6.245336  -6.246212  -6.2113914 -6.20858  ]
log_var_disp.mu [0.14517866 0.16518378 0.22705211 0.19399278 0.22105102 0.21622285
 0.15554336 0.14004068 0.16220586 0.22486557]
log_var_mean.sigma [0.03828404 0.028723   0.04509141 0.02029763 0.01978435 0.03146413
 0.03488404 0.02574086 0.03059431 0.05757351]
log_var_disp.sigma [0.04476827 0.04720033 0.04297895 0.06293873 0.05124351 0.04453851
 0.04400106 0.04900048 0.04351211 0.0499346 ]
perturb_mean_lfc.mu [[-0.07006194  0.10264269  0.05588979  0.01425817  0.02332943  0.00722477
   0.06275817  0.04462365 -0.01897361  0.04755367]
 [-0.11058009  0.06639721  0.04694334  0.03128977 -0.01657384  0.08505069
   0.06078687 -0.02399641 -0.07841256  0.03910826]
 [-0.00981476 -0.01720125 -0.03777379  0.07391844 -0.01008514 -0.07890587
  -0.1184442  -0.01291898 -0.06596126  0.03547969]
 [-0.04343232  0.04350057  0.0379506   0.05222842  0.05392796 -0.05830814
   0.0217